# CICDDoS2019 first-N flow feature extraction with cached CSV labels

This notebook avoids reloading the official CICDDoS2019 CSV label files every time you process a PCAP. Run the **Load official CSV labels** cell once, then process as many PCAP files as you want using the cached `labeler` object.

Keep `extract_first_n_flow_features_streaming_labeled.py` in the same directory as this notebook, or set `SCRIPT_PATH` to its location.

In [14]:
from pathlib import Path
import sys
import pandas as pd

# Change this if the .py file is somewhere else.
SCRIPT_PATH = Path("extract_first_n_flow_features_streaming_labeled.py").resolve()

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Cannot find script: {SCRIPT_PATH}")

sys.path.insert(0, str(SCRIPT_PATH.parent))

import extract_first_n_flow_features_streaming_labeled as extractor

print("Loaded module from:", extractor.__file__)

Loaded module from: /home/ubuntu/DDoS_ML/preprocessing/extract_first_n_flow_features_streaming_labeled.py


In [15]:
import importlib
import extract_first_n_flow_features_streaming_labeled as extractor

extractor = importlib.reload(extractor)

print("Reloaded:", extractor.__file__)

Reloaded: /home/ubuntu/DDoS_ML/preprocessing/extract_first_n_flow_features_streaming_labeled.py


## Configuration

Set the dataset paths here. For CICDDoS2019, the CSV files are usually inside one extra subdirectory, for example `CSV-03-11/03-11` and `CSV-01-12/01-12`.

In [6]:
DATASET_ROOT = Path("/home/ubuntu/datasets/CICDDoS2019")

# Choose the day you are processing.
DAY = "03-11"   # "03-11" or "01-12"

if DAY == "03-11":
    PCAP_DIR = DATASET_ROOT / "PCAP-03-11"
    LABEL_CSV_DIR = DATASET_ROOT / "CSV-03-11" / "03-11"
elif DAY == "01-12":
    # There are multiple 01-12 PCAP folders. This root lets us find all of them.
    PCAP_DIR = DATASET_ROOT
    LABEL_CSV_DIR = DATASET_ROOT / "CSV-01-12" / "01-12"
else:
    raise ValueError("DAY must be '03-11' or '01-12'")

OUTPUT_DIR = Path("./features_out") / DAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_PACKETS = 3
FLOW_TIMEOUT = 120.0
CSV_TIME_TOLERANCE = 30.0
CSV_TIME_OFFSET = 10800.0
BIDIRECTIONAL = False   # Start with False for easier CSV matching.
WRITE_INCOMPLETE_AT_END = False

print("PCAP_DIR:", PCAP_DIR)
print("LABEL_CSV_DIR:", LABEL_CSV_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

PCAP_DIR: /home/ubuntu/datasets/CICDDoS2019/PCAP-03-11
LABEL_CSV_DIR: /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11
OUTPUT_DIR: /home/ubuntu/DDoS_ML/preprocessing/features_out/03-11


## Load official CSV labels once

This is the slow step. Run it once per day. After it finishes, keep the kernel alive and process many PCAPs without reloading the CSV files.

In [17]:
labeler = extractor.CsvFlowLabeler(
    csv_dir=LABEL_CSV_DIR,
    tolerance_seconds=CSV_TIME_TOLERANCE,
    csv_time_offset_seconds=CSV_TIME_OFFSET,
    allow_reverse=True,
)
labeler.load()

print("Rows loaded:", f"{labeler.rows_loaded:,}")
print("Rows skipped:", f"{labeler.rows_skipped:,}")
print("Unique 5-tuples:", f"{len(labeler.index):,}")

Loading label CSV files from: /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11
Loaded labels from /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11/LDAP.csv total_rows_loaded=2,113,234 skipped=0
Loaded labels from /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11/MSSQL.csv total_rows_loaded=7,889,020 skipped=0
Loaded labels from /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11/NetBIOS.csv total_rows_loaded=11,344,919 skipped=0
Loaded labels from /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11/Portmap.csv total_rows_loaded=11,536,613 skipped=0
Loaded labels from /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11/Syn.csv total_rows_loaded=15,857,154 skipped=0
Loaded labels from /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11/UDP.csv total_rows_loaded=19,639,360 skipped=0
Loaded labels from /home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11/UDPLag.csv total_rows_loaded=20,364,525 skipped=0
Finished loading label index: files=7, rows_loaded=20,364,525, rows_skipped=0, unique_5tuples=17

## Check one flow

In [ ]:
flow_key = ("172.16.0.5", "192.168.50.4", 32768, 31143, "17")

records = labeler.index.get(flow_key)

print("records is None?", records is None)
print("Number of records:", len(records) if records else 0)

if records:
    print("Record type:", type(records[-1]))
    print("Last record:", records[-1])

    if isinstance(records[-1], dict):
        print("Record keys:", records[-1].keys())

records is None? False
Number of records: 2
Record type: <class 'tuple'>
Last record: (1541253686.890484, 'UDP', '/home/ubuntu/datasets/CICDDoS2019/CSV-03-11/03-11/UDP.csv')


## Process one PCAP

Use this cell to test one file first.

In [44]:
# Pass the exact PCAP filename here.
# You can include or omit ".pcap".
PCAP_NAME = "SAT-03-11-2018_0.pcap"

pcap_files = sorted(PCAP_DIR.rglob("*.pcap"))
print("Found PCAPs:", len(pcap_files))

matches = [
    p for p in pcap_files
    if p.name == PCAP_NAME or p.stem == PCAP_NAME
]

if not matches:
    raise FileNotFoundError(
        f"Could not find PCAP named {PCAP_NAME!r} under {PCAP_DIR}"
    )

if len(matches) > 1:
    print("Multiple matches found:")
    for p in matches:
        print(" ", p)
    raise ValueError("PCAP_NAME is ambiguous. Use the full filename or a more specific path.")

pcap_file = matches[0]
output_file = OUTPUT_DIR / f"{pcap_file.stem}_features_n{N_PACKETS}.csv"

print("Selected PCAP:", pcap_file)
print("Output file:", output_file)

Found PCAPs: 146
Selected PCAP: /home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/SAT-03-11-2018_0.pcap
Output file: features_out/03-11/SAT-03-11-2018_0_features_n3.csv


In [45]:
extractor.process_pcap(
    pcap_path=pcap_file,
    output_file=output_file,
    n_packets=N_PACKETS,
    label="unknown",
    bidirectional=BIDIRECTIONAL,
    flow_timeout=FLOW_TIMEOUT,
    progress_interval=100_000,
    cleanup_interval=100_000,
    flush_interval=100_000,
    write_incomplete_at_end=WRITE_INCOMPLETE_AT_END,
    labeler=labeler,
)

print("Wrote:", output_file)

Reading PCAP: /home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/SAT-03-11-2018_0.pcap
Writing CSV while reading: features_out/03-11/SAT-03-11-2018_0_features_n3.csv
Using first 3 packets per flow
Flow timeout: 120.0 seconds
Processed 100,000 packets... active_incomplete_flows=3,916, completed_cache=1,650, rows_written=1,730
Processed 200,000 packets... active_incomplete_flows=31,434, completed_cache=724, rows_written=2,088
Processed 300,000 packets... active_incomplete_flows=80,486, completed_cache=214, rows_written=2,116
Processed 400,000 packets... active_incomplete_flows=130,290, completed_cache=269, rows_written=2,171
Processed 500,000 packets... active_incomplete_flows=180,158, completed_cache=300, rows_written=2,203
Finished reading packets.
Total packets read: 511,063
IP packets used: 508,994
Rows written: 2,203
Label counts: {'unknown': 62, 'BENIGN': 1778, 'Portmap': 363}
Incomplete flows still in memory: 185,683
Completed-flow cache entries still in memory: 299
Expired incomplete f

### Check label results

Use this after processing a file to see whether labels are being matched.

In [28]:
df = pd.read_csv(output_file)
print("Rows:", len(df))
print("Label counts:")
print(df["label"].value_counts(dropna=False).head(30))
print("Label source counts:")
print(df["label_source"].value_counts(dropna=False).head(30))

df.head()

Rows: 2203
Label counts:
label
BENIGN     1778
Portmap     363
unknown      62
Name: count, dtype: int64
Label source counts:
label_source
csv:Portmap.csv    2141
csv_no_match         62
Name: count, dtype: int64


,src_ip,dst_ip,src_port,dst_port,protocol,flow_start_time,flow_end_time,packet_count,duration,total_bytes,...,iat_std,tcp_syn_count,tcp_ack_count,tcp_fin_count,tcp_rst_count,tcp_psh_count,tcp_urg_count,label,label_source,csv_match_time_diff
0,192.168.50.254,224.0.0.5,0,0,89,2018-11-03 12:18:16.964447,2018-11-03 12:18:16.964455,3,0.000008,282,...,0.000003,0,0,0,0,0,0,unknown,csv_no_match,NaN
1,192.168.50.253,224.0.0.5,0,0,89,2018-11-03 12:18:18.506537,2018-11-03 12:18:18.506545,3,0.000008,282,...,0.000003,0,0,0,0,0,0,unknown,csv_no_match,NaN
2,192.168.50.6,172.217.9.238,54805,80,6,2018-11-03 12:18:18.626325,2018-11-03 12:18:18.667575,3,0.041250,180,...,0.029164,0,3,2,0,0,0,BENIGN,csv:Portmap.csv,0.0
3,192.168.50.6,172.217.10.98,54801,443,6,2018-11-03 12:18:18.610581,2018-11-03 12:18:18.672988,3,0.062407,260,...,0.044066,0,3,0,0,2,0,BENIGN,csv:Portmap.csv,0.0
4,192.168.50.6,172.217.7.2,54800,443,6,2018-11-03 12:18:18.610579,2018-11-03 12:18:18.672990,3,0.062411,260,...,0.044130,0,3,0,0,2,0,BENIGN,csv:Portmap.csv,0.0


## Process all PCAP

### Step 1: Define the directory you want to process

In [7]:
from pathlib import Path

# Choose the PCAP directory you want to process
PCAP_BATCH_DIR = DATASET_ROOT / "PCAP-03-11"

# For 01-12, you could use one folder:
# PCAP_BATCH_DIR = DATASET_ROOT / "PCAP-01-12_0-0249"

# Or later, you can process all 01-12 split folders separately.

BATCH_OUTPUT_DIR = OUTPUT_DIR / DAY / "batch"
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pcap_files = sorted(PCAP_BATCH_DIR.rglob("*.pcap"))

print("PCAP directory:", PCAP_BATCH_DIR)
print("Output directory:", BATCH_OUTPUT_DIR)
print("PCAP files found:", len(pcap_files))

# for p in pcap_files[:10]:
#     print(p)

PCAP directory: /home/ubuntu/datasets/CICDDoS2019/PCAP-03-11
Output directory: features_out/03-11/03-11/batch
PCAP files found: 146


### Step 2: Add a small test batch first

In [35]:
TEST_LIMIT = 3

for i, pcap_file in enumerate(pcap_files[:TEST_LIMIT], start=1):
    output_file = BATCH_OUTPUT_DIR / f"{pcap_file.stem}_features_n{N_PACKETS}.csv"

    print("=" * 80)
    print(f"[{i}/{TEST_LIMIT}] Processing:", pcap_file)
    print("Output:", output_file)

    extractor.process_pcap(
        pcap_path=pcap_file,
        output_file=output_file,
        n_packets=N_PACKETS,
        label="unknown",
        bidirectional=BIDIRECTIONAL,
        flow_timeout=FLOW_TIMEOUT,
        progress_interval=100_000,
        cleanup_interval=100_000,
        flush_interval=100_000,
        write_incomplete_at_end=WRITE_INCOMPLETE_AT_END,
        labeler=labeler,
    )

    print("Wrote:", output_file)

[1/3] Processing: /home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/SAT-03-11-2018_0.pcap
Output: features_out/03-11/03-11/batch/SAT-03-11-2018_0_features_n3.csv
Reading PCAP: /home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/SAT-03-11-2018_0.pcap
Writing CSV while reading: features_out/03-11/03-11/batch/SAT-03-11-2018_0_features_n3.csv
Using first 3 packets per flow
Flow timeout: 120.0 seconds
Processed 100,000 packets... active_incomplete_flows=3,916, completed_cache=1,650, rows_written=1,730
Processed 200,000 packets... active_incomplete_flows=31,434, completed_cache=724, rows_written=2,088
Processed 300,000 packets... active_incomplete_flows=80,486, completed_cache=214, rows_written=2,116
Processed 400,000 packets... active_incomplete_flows=130,290, completed_cache=269, rows_written=2,171
Processed 500,000 packets... active_incomplete_flows=180,158, completed_cache=300, rows_written=2,203
Finished reading packets.
Total packets read: 511,063
IP packets used: 508,994
Rows written: 2,203
Label

### Step 3: Process all PCAPs under the directory

In [36]:
import traceback
import pandas as pd

SKIP_EXISTING = True

batch_results = []
failed_files = []

total_files = len(pcap_files)

for i, pcap_file in enumerate(pcap_files, start=1):
    output_file = BATCH_OUTPUT_DIR / f"{pcap_file.stem}_features_n{N_PACKETS}.csv"

    print("=" * 100)
    print(f"[{i}/{total_files}] Processing: {pcap_file.name}")
    print("Input: ", pcap_file)
    print("Output:", output_file)

    if SKIP_EXISTING and output_file.exists():
        print("Skipping because output already exists.")
        batch_results.append({
            "pcap_file": str(pcap_file),
            "output_file": str(output_file),
            "status": "skipped_existing",
        })
        continue

    try:
        extractor.process_pcap(
            pcap_path=pcap_file,
            output_file=output_file,
            n_packets=N_PACKETS,
            label="unknown",
            bidirectional=BIDIRECTIONAL,
            flow_timeout=FLOW_TIMEOUT,
            progress_interval=100_000,
            cleanup_interval=100_000,
            flush_interval=100_000,
            write_incomplete_at_end=WRITE_INCOMPLETE_AT_END,
            labeler=labeler,
        )

        batch_results.append({
            "pcap_file": str(pcap_file),
            "output_file": str(output_file),
            "status": "done",
        })

    except Exception as e:
        print("FAILED:", pcap_file)
        print(e)
        traceback.print_exc()

        failed_files.append(str(pcap_file))

        batch_results.append({
            "pcap_file": str(pcap_file),
            "output_file": str(output_file),
            "status": "failed",
            "error": str(e),
        })

print("=" * 100)
print("Batch complete.")
print("Total files:", total_files)
print("Failed files:", len(failed_files))

batch_results_df = pd.DataFrame(batch_results)
batch_results_df

[1/146] Processing: SAT-03-11-2018_0.pcap
Input:  /home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/SAT-03-11-2018_0.pcap
Output: features_out/03-11/03-11/batch/SAT-03-11-2018_0_features_n3.csv
Reading PCAP: /home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/SAT-03-11-2018_0.pcap
Writing CSV while reading: features_out/03-11/03-11/batch/SAT-03-11-2018_0_features_n3.csv
Using first 3 packets per flow
Flow timeout: 120.0 seconds
Processed 100,000 packets... active_incomplete_flows=3,916, completed_cache=1,650, rows_written=1,730
Processed 200,000 packets... active_incomplete_flows=31,434, completed_cache=724, rows_written=2,088
Processed 300,000 packets... active_incomplete_flows=80,486, completed_cache=214, rows_written=2,116
Processed 400,000 packets... active_incomplete_flows=130,290, completed_cache=269, rows_written=2,171
Processed 500,000 packets... active_incomplete_flows=180,158, completed_cache=300, rows_written=2,203
Finished reading packets.
Total packets read: 511,063
IP packets used: 5

,pcap_file,output_file,status
0,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done
1,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done
2,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done
3,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done
4,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done
...,...,...,...
141,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done
142,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done
143,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done
144,/home/ubuntu/datasets/CICDDoS2019/PCAP-03-11/S...,features_out/03-11/03-11/batch/SAT-03-11-2018_...,done


### Step 4: Save a batch log

In [37]:
batch_log_file = BATCH_OUTPUT_DIR / f"batch_log_n{N_PACKETS}.csv"
batch_results_df.to_csv(batch_log_file, index=False)

print("Batch log written to:", batch_log_file)

Batch log written to: features_out/03-11/03-11/batch/batch_log_n3.csv


### Step 5: Check label counts across all generated outputs

In [38]:
import pandas as pd
from collections import Counter

label_counter = Counter()
label_source_counter = Counter()
row_counter = 0

feature_files = sorted(BATCH_OUTPUT_DIR.glob(f"*_features_n{N_PACKETS}.csv"))

print("Feature files found:", len(feature_files))

for feature_file in feature_files:
    df = pd.read_csv(feature_file, usecols=["label", "label_source"])
    row_counter += len(df)
    label_counter.update(df["label"].astype(str))
    label_source_counter.update(df["label_source"].astype(str))

print("Total extracted rows:", row_counter)

print("\nLabel counts:")
for label, count in label_counter.most_common(30):
    print(label, count)

print("\nLabel source counts:")
for source, count in label_source_counter.most_common(30):
    print(source, count)

Feature files found: 146
Total extracted rows: 2769585

Label counts:
UDP 1941505
Syn 754745
BENIGN 23744
unknown 20747
MSSQL 19417
NetBIOS 6628
LDAP 2286
Portmap 363
UDPLag 150

Label source counts:
csv:UDP.csv 1886705
csv:Syn.csv 663918
csv:UDPLag.csv 162628
csv:MSSQL.csv 20968
csv_no_match 20747
csv:NetBIOS.csv 7414
csv:LDAP.csv 5064
csv:Portmap.csv 2141


### Step 6: Combine all feature CSVs into one file

In [8]:
import pandas as pd
from pathlib import Path

batch_dir = Path("/home/ubuntu/DDoS_ML/preprocessing/features_out/03-11/03-11/batch")
feature_files = sorted(batch_dir.glob(f"*_features_n{N_PACKETS}.csv"))

print(f"Found {len(feature_files)} feature CSV files to combine.")

combined_df = pd.concat(
    (pd.read_csv(f) for f in feature_files),
    ignore_index=True,
)

combined_output = batch_dir / f"combined_features_n{N_PACKETS}.csv"
combined_df.to_csv(combined_output, index=False)

print(f"Combined shape: {combined_df.shape}")
print(f"Written to: {combined_output}")
combined_df["label"].value_counts(dropna=False)

Found 146 feature CSV files to combine.


Combined shape: (2769585, 29)
Written to: /home/ubuntu/DDoS_ML/preprocessing/features_out/03-11/03-11/batch/combined_features_n3.csv


label
UDP        1941505
Syn         754745
BENIGN       23744
unknown      20747
MSSQL        19417
NetBIOS       6628
LDAP          2286
Portmap        363
UDPLag         150
Name: count, dtype: int64

### Generate summary

In [39]:
from collections import Counter
from pathlib import Path
import pandas as pd

# Directory where your per-PCAP feature CSVs are written
SUMMARY_INPUT_DIR = BATCH_OUTPUT_DIR

# Output text summary file
summary_txt_file = SUMMARY_INPUT_DIR / f"pcap_feature_summary_n{N_PACKETS}.txt"

feature_files = sorted(SUMMARY_INPUT_DIR.glob(f"*_features_n{N_PACKETS}.csv"))

print("Feature CSVs found:", len(feature_files))
print("Summary will be written to:", summary_txt_file)

Feature CSVs found: 146
Summary will be written to: features_out/03-11/03-11/batch/pcap_feature_summary_n3.txt


In [40]:
overall_label_counts = Counter()
overall_label_source_counts = Counter()
overall_rows = 0

summary_lines = []

summary_lines.append("CICDDoS2019 PCAP Feature Extraction Summary")
summary_lines.append("=" * 80)
summary_lines.append(f"Input feature directory: {SUMMARY_INPUT_DIR}")
summary_lines.append(f"Number of feature CSV files: {len(feature_files)}")
summary_lines.append(f"N_PACKETS: {N_PACKETS}")
summary_lines.append(f"FLOW_TIMEOUT: {FLOW_TIMEOUT}")
summary_lines.append(f"BIDIRECTIONAL: {BIDIRECTIONAL}")
summary_lines.append(f"WRITE_INCOMPLETE_AT_END: {WRITE_INCOMPLETE_AT_END}")
summary_lines.append(f"CSV_TIME_TOLERANCE: {CSV_TIME_TOLERANCE}")
summary_lines.append(f"CSV_TIME_OFFSET: {CSV_TIME_OFFSET}")
summary_lines.append("")

per_file_rows = []

for feature_file in feature_files:
    try:
        df = pd.read_csv(feature_file)

        rows = len(df)
        overall_rows += rows

        label_counts = df["label"].astype(str).value_counts(dropna=False)
        label_source_counts = df["label_source"].astype(str).value_counts(dropna=False)

        overall_label_counts.update(df["label"].astype(str))
        overall_label_source_counts.update(df["label_source"].astype(str))

        unknown_count = int((df["label"].astype(str) == "unknown").sum())
        unknown_pct = (unknown_count / rows * 100) if rows > 0 else 0.0

        per_file_rows.append({
            "feature_file": feature_file.name,
            "rows": rows,
            "unknown": unknown_count,
            "unknown_pct": unknown_pct,
        })

        summary_lines.append("-" * 80)
        summary_lines.append(f"File: {feature_file.name}")
        summary_lines.append(f"Rows: {rows:,}")
        summary_lines.append(f"Unknown labels: {unknown_count:,} ({unknown_pct:.2f}%)")

        summary_lines.append("Label counts:")
        for label, count in label_counts.items():
            summary_lines.append(f"  {label}: {count:,}")

        summary_lines.append("Label source counts:")
        for source, count in label_source_counts.items():
            summary_lines.append(f"  {source}: {count:,}")

        summary_lines.append("")

    except Exception as e:
        summary_lines.append("-" * 80)
        summary_lines.append(f"File: {feature_file.name}")
        summary_lines.append(f"ERROR: {e}")
        summary_lines.append("")

In [41]:
summary_lines.append("=" * 80)
summary_lines.append("Overall Summary")
summary_lines.append("=" * 80)
summary_lines.append(f"Total feature CSV files: {len(feature_files):,}")
summary_lines.append(f"Total extracted rows: {overall_rows:,}")
summary_lines.append("")

summary_lines.append("Overall label counts:")
for label, count in overall_label_counts.most_common():
    pct = (count / overall_rows * 100) if overall_rows > 0 else 0.0
    summary_lines.append(f"  {label}: {count:,} ({pct:.2f}%)")

summary_lines.append("")
summary_lines.append("Overall label source counts:")
for source, count in overall_label_source_counts.most_common():
    pct = (count / overall_rows * 100) if overall_rows > 0 else 0.0
    summary_lines.append(f"  {source}: {count:,} ({pct:.2f}%)")

summary_lines.append("")
summary_lines.append("Files with highest unknown percentage:")
per_file_df = pd.DataFrame(per_file_rows)

if not per_file_df.empty:
    worst = per_file_df.sort_values("unknown_pct", ascending=False).head(20)

    for row in worst.itertuples(index=False):
        summary_lines.append(
            f"  {row.feature_file}: "
            f"{row.unknown:,}/{row.rows:,} unknown "
            f"({row.unknown_pct:.2f}%)"
        )

summary_txt_file.write_text("\n".join(summary_lines), encoding="utf-8")

print("Summary written to:", summary_txt_file)

Summary written to: features_out/03-11/03-11/batch/pcap_feature_summary_n3.txt
